# Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Conv2d, BatchNorm2d, SiLU, Identity, Module, Sequential
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
import random
import math
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"

# U-Net implementation

In [5]:
class Conv(Sequential):
    def __init__(self, cin, cout, k=3, norm=True, act="silu"):
        super().__init__(
        BatchNorm2d(cin) if norm else Identity(),
        Conv2d(cin, cout, k, padding="same"),
        {"none": Identity(), "silu":SiLU()}[act]
        )
        
class DoubleConv(Sequential):
    def __init__(self, cin, cout, k=3, norm=True, act="silu"):
        super().__init__(
            Conv(cin, cout, k, norm, act),
            Conv(cout, cout, k, norm, act)
        )

In [6]:
class UNET(Module):
    def __init__(self, c):
        super().__init__()
        self.d1 = DoubleConv(2, c)
        self.d2 = DoubleConv(c, c*2)
        self.d3 = DoubleConv(c*2, c*4)
        self.d4 = DoubleConv(c*4, c*8)

        self.bn = DoubleConv(c*8, c*8)

        self.u4 = DoubleConv(c*16, c*4)
        self.u3 = DoubleConv(c*8, c*2)
        self.u2 = DoubleConv(c*4, c)
        self.u1 = DoubleConv(c*2, c)
        self.u0 = DoubleConv(c, 1)

    def forward(self, x):
        x = F.pad(x, [2,2,2,2])
        x1 = F.max_pool2d(self.d1(x), 2, 2)
        x2 = F.max_pool2d(self.d2(x1), 2, 2)
        x3 = F.max_pool2d(self.d3(x2), 2, 2)
        x4 = F.max_pool2d(self.d4(x3), 2, 2)

        x = self.bn(x4)

        x = F.interpolate(self.u4(torch.cat([x4, x], dim=1)), scale_factor=2)
        x = F.interpolate(self.u3(torch.cat([x3, x], dim=1)), scale_factor=2)
        x = F.interpolate(self.u2(torch.cat([x2, x], dim=1)), scale_factor=2)
        x = F.interpolate(self.u1(torch.cat([x1, x], dim=1)), scale_factor=2)
        x = self.u0(x)
        x = F.pad(x, [-2,-2,-2,-2])
        return x

# MNIST setup

In [7]:
def get_mnist_dataloader(batch_size=64):
    train_dataset = datasets.MNIST(
        root='./data',
        train=True,
        download=True,
        transform=ToTensor()
    )
    train_loader = DataLoader(
        dataset=train_dataset, 
        batch_size=batch_size, 
        shuffle=True,
    )
    return train_loader

if __name__ == "__main__":
    train_loader = get_mnist_dataloader(batch_size=64)
    
    images, labels = next(iter(train_loader))
    print(f"Feature batch shape: {images.shape}")
    print(f"Labels batch shape: {labels.shape}")

Feature batch shape: torch.Size([64, 1, 28, 28])
Labels batch shape: torch.Size([64])


# Defining model

In [8]:
save_path = "flow-matching-basics.pth"
unet = UNET(16).to(device)
try:
   state_dict = torch.load(save_path, map_location=device) 
   unet.load_state_dict(state_dict)
except:
   pass

optim = torch.optim.Adam(unet.parameters(), fused=True)

c:\Users\liquidcat\anaconda3\envs\flow\Lib\site-packages\torch\cuda\__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5070 with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeForce RTX 5070 GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(
C:\Users\liquidcat\AppData\Local\Temp\ipykernel_21832\782286533.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that

# Training

In [9]:
def train_epoch(unet, loader, optimizer, epoch):
    unet.train()
    losses = []
    for x, _ in loader:
        x = x.to(device)

        eps = torch.randn_like(x)
        t = torch.rand(len(), 1, 1, 1, device=device)
        z = x * (1 - t) + eps * t
        y = eps - x
        zt = torch.cat((z, t.repeat(1, 1, 28, 28)), dim=1).to(device)
        y_hat = unet(zt)

        loss = (y - y_hat).pow(2).mean()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        losses.append(loss.item())

    plt.figure(None, (5,1))
    plt.xticks([])
    plt.title(f"epoch {epoch+1}")
    plt.plot(losses)
    plt.show()

    torch.save(unet.state_dict(), save_path)
    print(f"Model successfully saved to {save_path}")

for g in optim.param_groups:
    g['lr'] = 67e-4 / 2 # 67

for epoch in range(100):
    train_epoch(unet, train_loader, optim, epoch)

RuntimeError: CUDA error: no kernel image is available for execution on the device
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


# Interference

In [ ]:
def visualize_results(unet, num_samples=16, steps=20):
    unet.eval()
    x_t = torch.randn((num_samples, 1, 28, 28), device=device)
    ts = torch.linspace(1.0, 0.0, steps + 1, device=device)
    n = math.sqrt(num_samples)